# Model Training — Baseline Run

Baseline using raw features only (no engineering) to establish a comparison point before adding engineered features.

## 1. Setup

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import PrecisionRecallDisplay, RocCurveDisplay

from src.data import load_config, load_raw, clean, split
from src.model import train_baseline, evaluate

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 4)

cfg = load_config("../config.yaml")

## 2. Load & Prepare Data

In [ ]:
df = clean(load_raw("../data/raw"))
train_df, test_df = split(df, test_size=cfg["data"]["test_size"], random_state=cfg["split"]["random_state"])

TARGET   = cfg["data"]["target_col"]
DROP_COLS = ["id", TARGET]

X_train = train_df.drop(columns=DROP_COLS)
y_train = train_df[TARGET]
X_test  = test_df.drop(columns=DROP_COLS)
y_test  = test_df[TARGET]

print(f"Features: {X_train.columns.tolist()}")
print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")

## 3. Baseline Models

In [ ]:
baselines = train_baseline(X_train, y_train)

lr_probs  = baselines["logistic_regression"].predict_proba(X_test)[:, 1]
xgb_probs = baselines["xgboost_baseline"].predict_proba(X_test)[:, 1]

results = [
    evaluate("Logistic Regression", y_test, lr_probs),
    evaluate("XGBoost (baseline)",  y_test, xgb_probs),
]

results_df = pd.DataFrame(results).set_index("model")
results_df

## 4. Precision-Recall & ROC Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models = [
    ("Logistic Regression", lr_probs),
    ("XGBoost (baseline)",  xgb_probs),
]

for name, probs in models:
    PrecisionRecallDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[0]
    )
    RocCurveDisplay.from_predictions(
        y_test, probs, name=name, ax=axes[1]
    )

# Baseline: random classifier PR line
axes[0].axhline(y_test.mean(), color="grey", linestyle="--", label=f"Random ({y_test.mean():.2f})")
axes[0].set_title("Precision-Recall Curve")
axes[1].set_title("ROC Curve")
axes[0].legend()
axes[1].legend()
plt.tight_layout()

## 5. XGBoost Feature Importance

In [ ]:
xgb_model   = baselines["xgboost_baseline"]
importances = pd.Series(xgb_model.feature_importances_, index=X_train.columns).sort_values()

plt.figure(figsize=(8, 5))
colors = ["#DD8452" if v == importances.max() else "#4C72B0" for v in importances.values]
plt.barh(importances.index, importances.values, color=colors)
plt.title("XGBoost Feature Importance (baseline)")
plt.xlabel("Importance score")
plt.tight_layout()

## 6. Baseline Summary

In [ ]:
print("=" * 45)
print("BASELINE RESULTS (raw features, no tuning)")
print("=" * 45)
print(results_df.to_string())
print()
print("Next steps:")
print("  1. Re-run with engineered features (features.py) — measure PR-AUC lift")
print("  2. Tune XGBoost with Optuna (tune_hyperparameters)")
print("  3. SHAP feature selection (select_features_shap)")
print("  4. Save final model (save_model)")